# 1. Import and Hardware Setup

In [76]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import time
import copy

!pip install tqdm -q
from tqdm.auto import tqdm

!pip install wandb -q
import wandb

In [77]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [78]:
DATA_PATH = './Data'
SAVE_PATH = './Model'

In [79]:
# Login Weight & Bias
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

# 2. Hyperparameters

In [80]:
BATCH_SIZE = 32
IMG_SIZE = 227
IMG_CHANNELS = 3
NUM_CLASSES = 101

EPOCHS = 500
LR = 1e-4
DROPOUT_RATE = 0.5

# 3. Dataset Preparation

In [81]:
stats = ((0.545, 0.443, 0.344), (0.269, 0.271, 0.276))

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(227),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

test_transform = transforms.Compose([
    transforms.Resize((227, 227)),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

In [82]:
# Load the full train data
full_train_data = datasets.Food101(root=DATA_PATH, download=True, split='train', transform=train_transform)

# Split the full train data into train and validation data
train_size = int(0.8 * len(full_train_data))
val_size = len(full_train_data) - train_size

train_subset, val_subset = random_split(full_train_data, (train_size, val_size))

# Change transform von val_subset to test_transform
val_subset.dataset = copy.copy(full_train_data)
val_subset.dataset.transform = test_transform

# Load Test Dataset
test_dataset = datasets.Food101(root=DATA_PATH, download=True, split='test', transform=test_transform)

In [83]:
# Loader
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True, persistent_workers=True)

val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True, persistent_workers=True)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True, persistent_workers=True)



# 4. AlexNet Architecture

![AlexNet](AlexNet.png)

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, img_size, in_channels, num_classes, dropout_rate):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            # Conv 1: 227x227 -> 55x55
            nn.Conv2d(in_channels, 96, kernel_size=11, stride=4),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=0.0001, beta=0.75, k=2),

            # MaxPool 1: 55x55 -> 27x27
            nn.MaxPool2d(kernel_size=3, stride=2),

            # Conv 2: 27x27 -> 27x27
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=0.0001, beta=0.75, k=2),

            # MaxPool 2: 27x27 -> 13x13
            nn.MaxPool2d(kernel_size=3, stride=2),

            # Conv 3: 13x13 -> 13x13
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv 4: 13x13 -> 13x13
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv 5: 13x13 -> 13x13
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # MaxPool 3: 13x13 -> 6x6
            nn.MaxPool2d(kernel_size=3, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(9216, 4096), #  6x6x256 = 9216 -> 4096
            nn.ReLU(inplace=True),

            nn.Dropout(dropout_rate),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),

            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.flatten(x, 1)
        logits = self.classifier(x)

        return logits

model = AlexNet(IMG_SIZE, IMG_CHANNELS, NUM_CLASSES, DROPOUT_RATE).to(device)

# 5. Training

In [85]:
class EarlyStopping:
    """ Early stops the training if validation loss doesn't improve after 
        a given patience.
    """
    def __init__(self, patience=5, verbose=False, delta=0, save_path='best_checkpoint.pth', 
                                            trace_func=print):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.save_path = save_path
        self.trace_func = trace_func
        
        self.counter = 0
        self.best_score = None
        self.val_loss_min = np.inf
        self.early_stop = False

    def __call__(self, val_loss, model):

        # 1. First epoch
        if self.best_score is None:
            self.best_score = val_loss
            self.save_checkpoint(val_loss, model)

        # 2. If the validation loss is larger than the best score
        # -> Increase the counter
        elif val_loss > self.best_score - self.delta:
            self.counter += 1
            self.trace_func(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

        # 3. The Validation Loss reduced -> Save the model
        else:
            self.best_score = val_loss
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        self.val_loss_min = val_loss
        torch.save(model.state_dict(), self.save_path)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.1)

# Using GradScaler to prevent Gradient Underflow when using FP16
scaler = torch.amp.GradScaler(device)

In [87]:
def train(model, loader, criterion, optimizer, scheduler):
    # Set the model into train mode
    model.train()
    
    # Define the loss and acc
    train_loss, correct = 0, 0

    # Create Loading Progress with
    loop = tqdm(loader, desc="Training", leave=False)

    # Loop through the loop
    for x, y in loop:
        # Move training data to device
        x, y = x.to(device), y.to(device)

        # Clear the gradients of last batch, set to None to save memory
        optimizer.zero_grad(set_to_none=True)

        # 
        with torch.amp.autocast(device_type=device.type):
            # get the prediction
            out = model(x)
            # Calculate loss
            loss = criterion(out, y)

        # Scaling the loss -> Backpropagation
        scaler.scale(loss).backward()
        # scaler.scale(loss): multiplies the loss by a larger "scale factor"
        # -> The loss becomes larger, moving them out of the "underflow zone"
        # .backward(): performs the standard backpropagation

        # 
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Unscale the gradients before the optimizer updates the weights
        scaler.step(optimizer)

        # Adjusts the scale factor for the next batch
        scaler.update()

        train_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()

    return train_loss / len(loader.dataset), correct / len(loader.dataset)

In [88]:
def validate(model, loader, criterion):
    model.eval()
    val_loss, val_acc = 0, 0

    loop = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for x, y in loop:
            x, y = x.to(device), y.to(device)

            out = model(x)
            loss = criterion(out, y)

            val_loss += loss.item() * x.size(0)
            val_acc += (out.argmax(1) == y).sum().item()

    return val_loss / len(loader.dataset), val_acc / len(loader.dataset)

In [89]:
def evaluate(model, loader):
    model.eval()
    eval_acc = 0

    loop = tqdm(loader, desc="Testing", leave=False)

    with torch.no_grad():
        for x, y in loop:
            x, y = x.to(device), y.to(device)
            out = model(x)

            eval_acc += (out.argmax(1) == y).sum().item()

    return eval_acc / len(loader.dataset)

In [ ]:
import numpy as np

wandb.init(
    project = "AlexNet",
    config = {
        "Architecture": "AlexNet",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMG_SIZE
    }
)

train_accuracies, test_accuracies = [], []
train_losses, test_losses = [], []
early_stopping = EarlyStopping(patience=3, save_path='AlexNet_best_model.pth')

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, scheduler)
    val_loss, val_acc = validate(model, val_loader, criterion)
    scheduler.step(val_loss)
    
    test_acc = evaluate(model, test_loader)

    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    train_losses.append(train_loss)
    test_losses.append(val_loss)

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "lr": optimizer.param_groups[0]['lr']
    })

    print(f"{epoch+1}/{EPOCHS}: train_loss: {train_loss:.4f}, val_loss: {val_loss:.4f}," +
                             f" train_acc: {train_acc:.4f}, val_acc: {val_acc:.4f}, test_acc: {test_acc:.4f}")

    early_stopping(val_loss, model)

    if early_stopping.early_stop:
        print("Early Stopping")
        break



wandb.finish()

epoch,▁█
lr,▁▁
test_acc,▁▁
train_acc,▁▁
val_acc,▁▁
val_loss,▁▁
+1,...
epoch,2
lr,0.01
test_acc,0.0099
train_acc,0.01013


Training:   0%|          | 0/1894 [00:00<?, ?it/s]

Validation:   0%|          | 0/474 [00:00<?, ?it/s]

Testing:   0%|          | 0/790 [00:00<?, ?it/s]

1/500: train_loss: 4.5958, val_loss: 4.5369, train_acc: 0.0135, val_acc: 0.0185, test_acc: 0.0194


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79bad8936de0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79bad8936de0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Validation:   0%|          | 0/474 [00:00<?, ?it/s]

Testing:   0%|          | 0/790 [00:00<?, ?it/s]

3/500: train_loss: 4.5208, val_loss: 4.4635, train_acc: 0.0195, val_acc: 0.0213, test_acc: 0.0229


Training:   0%|          | 0/1894 [00:00<?, ?it/s]

# 6. Result Plot

In [ ]:
plt.figure(figuresize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.title('Loss History')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(test_accuracies, label='Test Accuracy')
plt.title('Accuracy History')
plt.legend()

# 7. Heatmap with Grad-CAM

In [ ]:
import torchvision.transforms.functional as F_vision
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import random

# 1. Setup the Grad-CAM object
# AlexNet's last conv layer is the 3. to last element in the
# feature_extractor (followed by ReLU and MaxPool)
target_layers = [model.feature_extractor[-3]]
cam = GradCAM(model=model, target_layers=target_layers)

# 2. Pick a random image from the test set
imgs, labels = next(iter(test_loader))
indx = random.randint(0, len(imgs) - 1)
input_tensor = imgs[indx].squeeze(0).to(device)
label = labels[indx].item()

# 3. Generate the Grad-CAM heatmap
grayscale_cam = cam(input_tensor=input_tensor, targets=None)
grayscale_cam = grayscale_cam[0, :]

# 4. Prepare the image for Visualization
mean = torch.tensor(stats[0]).view(3, 1, 1)
std = torch.tensor(stats[1]).view(3, 1, 1)
rgb_img = imgs[indx] * std + mean # Denormalize
rgb_img = rgb_img.permute(1, 2, 0).numpy()
rgb_img = np.clip(rgb_img, 0, 1)

# 5. Overlay the heatmap on the image
visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

# 6. Get the Prediction
model.eval()
with torch.no_grad():
    output = model(input_tensor)
    pred = output.argmax(1).item()

# 7. Plotting
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title(f"Original (Class: {label})")
plt.imshow(rgb_img)
plt.axis('off')
plt.subplot(1, 2, 2)
plt.title(f"Grad-CAM (Pred: {pred})")
plt.imshow(visualization)
plt.axis('off')
plt.tight_layout()
plt.show()